In [34]:
# Import
import sionna.rt
import os

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [35]:
# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
xml_path = "/data/hw/sionna/ws/scenes/scenes_default/munich/munich.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")

try:
    scene = load_scene(xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
Heilig_Geist-itu_marble        | marble              
Heilig_Geist-itu_metal         | metal               
Frauenkirche-itu_marble        | marble              
Frauenkirche-itu_metal         | metal               
St__Peter-itu_marble           | marble              
St__Peter-itu_metal            | metal               
ground                         | concrete            
no-name-37                     | wood                
no-name-38                     | brick               
no-name-39                     | metal               
no-name-40                     | marble              

[정의된 재질 목록]
 - 이름: marble          (Type: marble, Thickness: [0.1])
 - 이름: metal           (Type: metal, Thickness: [0.1])
 - 이름: concrete        (Type: concrete, Thickness: [0.1])
 - 이름: wood            (Type: wood, Thickness: [0.1])
 - 이름: brick           (Type: brick, Thickness: [0.1])


In [36]:
# Import
import sionna.rt
import os

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [37]:
# 1) XML 파일 읽기
with open(xml_path, "r", encoding="utf-8") as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환
if not os.path.exists(meshes_dir):
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

abs_mesh_path = meshes_dir if meshes_dir.endswith("/") else meshes_dir + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 XML 저장
fixed_xml_path = xml_path.replace(".xml", "_abs.xml")
with open(fixed_xml_path, "w", encoding="utf-8") as f:
    f.write(xml_content_fixed)

# 4) 수정본으로 로드
try:
    scene = load_scene(fixed_xml_path)   # ★ 여기 중요
    print("[성공] 장면(Scene) 로드 완료:", fixed_xml_path)
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise

[성공] 장면(Scene) 로드 완료: /data/hw/sionna/ws/scenes/scenes_default/munich/munich_abs.xml


In [19]:
scene.preview(point_picker=True)

In [38]:
points_xyz = [
    (-644.788,  101.643, 0.000),
    (-491.239,   53.540, 0.000),
    (-382.395,   41.825, -0.000),
    (-227.403,  -21.641, 0.000),
    ( -80.882,  -85.347, -0.000),
]

# 도로 생성은 (x,y)만 쓰고, z는 road_z로 통일
points_xy = [(x,y) for (x,y,_) in points_xyz]


In [ ]:
import math

def make_road_rectangle_xml(seg_id, p0, p1, width=8.0, z=0.02):
    x0, y0 = p0
    x1, y1 = p1
    dx, dy = (x1-x0), (y1-y0)
    L = math.hypot(dx, dy)
    if L < 1e-6:
        return ""

    cx, cy = (x0+x1)/2, (y0+y1)/2
    theta_deg = math.degrees(math.atan2(dy, dx))

    return f"""
  <shape type="rectangle" id="road_{seg_id}">
    <transform name="to_world">
      <scale x="{width}" y="{L}" z="1"/>
      <rotate x="1" angle="-90"/>
      <rotate z="1" angle="{theta_deg}"/>
      <translate x="{cx}" y="{cy}" z="{z}"/>
    </transform>

    <!-- ✅ bsdf에 id를 반드시 넣어줘야 함 -->
    <bsdf type="diffuse" id="mat-road-{seg_id}">
      <rgb name="reflectance" value="0.1,0.1,0.1"/>
    </bsdf>
  </shape>
"""


In [40]:
# 예시: 너가 이미 만들었던 치환본 경로
# fixed_xml_path = "/data/hw/sionna/ws/scenes/scenes_default/munich/munich_abs.xml"

road_xml_path = fixed_xml_path.replace(".xml", "_roads.xml")

road_xml_path = add_roads_to_xml(
    xml_in_path=fixed_xml_path,
    xml_out_path=road_xml_path,
    points_xy=points_xy,
    road_width=8.0,   # 도로 폭(m) 원하는 값으로
    road_z=0.02       # z-fighting 방지로 살짝 띄움
)

scene_road = load_scene(road_xml_path, merge_shapes=False)
print("[완료] 도로 포함 씬 로드:", road_xml_path)

# 확인
scene_road.preview()


AttributeError: 'NoneType' object has no attribute 'startswith'

In [27]:
scene = load_scene(sionna.rt.scene.simple_street_canyon, merge_shapes=False)
scene.objects
floor = scene.get("floor")
print("Position (x,y,z) [m]: ", floor.position)
print("Orientation (alpha, beta, gamma) [rad]: ", floor.orientation)
print("Scaling: ", floor.scaling)
print("Velocity (x,y,z) [m/s]: ", floor.velocity)
floor.radio_material
scene.frequency = 28e9 # in Hz; implicitly updates RadioMaterials that implement frequency dependent properties
floor.radio_material # Note that the conductivity (sigma) changes automatically


Position (x,y,z) [m]:  [[-0.769669, 0.238537, -0.0307941]]
Orientation (alpha, beta, gamma) [rad]:  [[0, 0, 0]]
Scaling:  [[1, 1, 1]]
Velocity (x,y,z) [m/s]:  [[0, 0, 0]]


ITURadioMaterial type=concrete
                 eta_r=5.240
                 sigma=0.626
                 thickness=0.100
                 scattering_coefficient=0.000
                 xpd_coefficient=0.000

In [28]:
scene = load_scene(sionna.rt.scene.munich, merge_shapes=True) # Merge shapes to speed-up computations

# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="cross")

# Create transmitter
tx = Transmitter(name="tx",
                 position=[8.5,21,27],
                 display_radius=2)

# Add transmitter instance to scene
scene.add(tx)

# Create a receiver
rx = Receiver(name="rx",
              position=[45,90,1.5],
              display_radius=2)

# Add receiver instance to scene
scene.add(rx)

tx.look_at(rx) # Transmitter points towards receiver

In [29]:
p_solver  = PathSolver()

# Compute propagation paths
paths = p_solver(scene=scene,
                 max_depth=5,
                 los=True,
                 specular_reflection=True,
                 diffuse_reflection=False,
                 refraction=True,
                 synthetic_array=False,
                 seed=41)

In [25]:
scene.preview(point_picker=True)

In [ ]:
import math

def make_road_rectangle_xml(seg_id, p0, p1, width=8.0, z=0.02):
    x0, y0 = p0
    x1, y1 = p1
    dx, dy = (x1-x0), (y1-y0)
    L = math.hypot(dx, dy)
    if L < 1e-6:
        return ""

    cx, cy = (x0+x1)/2, (y0+y1)/2
    theta_deg = math.degrees(math.atan2(dy, dx))

    # 바닥에 눕히기: rectangle을 x축 -90도로 눕힌 뒤 z축 회전
    return f"""
  <shape type="rectangle" id="road_{seg_id}">
    <transform name="to_world">
      <scale x="{width}" y="{L}" z="1"/>
      <rotate x="1" angle="-90"/>
      <rotate z="1" angle="{theta_deg}"/>
      <translate x="{cx}" y="{cy}" z="{z}"/>
    </transform>
    <bsdf type="diffuse">
      <rgb name="reflectance" value="0.1,0.1,0.1"/>
    </bsdf>
  </shape>
"""

def add_roads_to_xml(xml_in_path, xml_out_path, points, road_width=8.0, road_z=0.02):
    with open(xml_in_path, "r", encoding="utf-8") as f:
        xml = f.read()

    roads_xml = ""
    for i in range(len(points)-1):
        roads_xml += make_road_rectangle_xml(i, points[i], points[i+1],
                                             width=road_width, z=road_z)

    xml2 = xml.replace("</scene>", roads_xml + "\n</scene>")

    with open(xml_out_path, "w", encoding="utf-8") as f:
        f.write(xml2)ene.preview(point_pic

    return xml_out_path


In [16]:
# fixed_xml_path: 네가 meshes 절대경로 치환한 munich_abs.xml
road_xml_path = fixed_xml_path.replace(".xml", "_roads.xml")

road_xml_path = add_roads_to_xml(
    xml_in_path=fixed_xml_path,
    xml_out_path=road_xml_path,
    points=points,
    road_width=8.0,   # 도로 폭(m)
    road_z=0.02       # 지면과 살짝 띄우기
)

print("도로 포함 XML 저장:", road_xml_path)


도로 포함 XML 저장: /data/hw/sionna/ws/scenes/scenes_default/munich/munich_abs_roads.xml


In [18]:
scene_road = load_scene(road_xml_path, merge_shapes=False)
print("도로 포함 씬 로드 완료")

if not no_preview:
    scene.preview();

도로 포함 씬 로드 완료


In [8]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

In [9]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
[실패] ID가 'elm__00'인 Shape를 mi_scene에서 찾을 수 없습니다.


논문 기반 Burst 데이터 생성(3 BS + 8x8 MIMO)

In [6]:
import numpy as np
import os

# 파일 목록
file_list = [
    'channel_history_burst_mimo_3bs_part0.npy',
    'channel_history_burst_mimo_3bs_part1.npy',
    'channel_history_burst_mimo_3bs_part2.npy',
    'channel_history_burst_mimo_3bs_part3.npy'
]

combined_data = []

print("🔄 파일 병합 시작...")
for fname in file_list:
    if os.path.exists(fname):
        data = np.load(fname, allow_pickle=True)
        # 리스트(Burst들)를 합침
        combined_data.extend(data) 
        print(f" - {fname} 로드 완료 ({len(data)}개 버스트)")
    else:
        print(f"⚠️ 파일 없음: {fname}")

# 최종 저장
final_name = 'channel_history_burst_mimo_3bs_final.npy'
np.save(final_name, combined_data)
print(f"✅ 병합 완료! 총 버스트 수: {len(combined_data)}")
print(f"💾 최종 파일: {final_name}")

🔄 파일 병합 시작...
 - channel_history_burst_mimo_3bs_part0.npy 로드 완료 (250개 버스트)
 - channel_history_burst_mimo_3bs_part1.npy 로드 완료 (250개 버스트)
 - channel_history_burst_mimo_3bs_part2.npy 로드 완료 (250개 버스트)
 - channel_history_burst_mimo_3bs_part3.npy 로드 완료 (250개 버스트)
✅ 병합 완료! 총 버스트 수: 1000
💾 최종 파일: channel_history_burst_mimo_3bs_final.npy


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
import os
import glob
from tqdm import tqdm

# ==============================================================================
# 1. 설정 파라미터
# ==============================================================================
# 데이터 파일 경로 패턴 (part0.npy, part1.npy ... 모두 읽음)
DATA_FILE_PATTERN = "channel_history_burst_mimo_3bs_part*.npy"

# 물리 계층 파라미터
BANDWIDTH = 15e6
NUM_SUBCARRIERS = 64
FREQUENCIES = np.linspace(-BANDWIDTH/2, BANDWIDTH/2, NUM_SUBCARRIERS)

# 학습 파라미터
INPUT_SEQ_LEN = 10   # 과거 10개 보고
PRED_SEQ_LEN = 10    # 미래 10개 예측
BATCH_SIZE = 64      # GPU 메모리에 따라 조절 (32 ~ 128)
EPOCHS = 100         # 학습 반복 횟수
VALIDATION_SPLIT = 0.2 # 검증 데이터 비율

# ==============================================================================
# 2. 모델 정의 (보내주신 코드 + 수정)
# ==============================================================================
def build_paper_cnn_lstm(input_steps=10, pred_steps=10):
    # 입력: (Time, Rx=8, Tx=24, Freq_RealImag=128)
    inputs = Input(shape=(input_steps, 8, 24, 128))
    
    # 1. Spatial Feature Extraction (CNN)
    x = layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.Conv2D(128, (3,3), padding='same', activation='relu'))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    
    # Pooling: (8, 24) -> (4, 12)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    
    # Flatten: (Time, 4*12*128)
    x = layers.TimeDistributed(layers.Flatten())(x)
    
    # 2. Temporal Feature Extraction (LSTM)
    x = layers.LSTM(512, return_sequences=False)(x) # Encoder
    
    x = layers.RepeatVector(pred_steps)(x)            # Future placeholder
    x = layers.LSTM(512, return_sequences=True)(x)    # Decoder
    
    # 3. Output Reconstruction
    # 원래 차원인 8*24*128 (Rx*Tx*Freq*2)로 복원
    output_dim = 8 * 24 * 128
    x = layers.TimeDistributed(layers.Dense(output_dim))(x)
    outputs = layers.Reshape((pred_steps, 8, 24, 128))(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Paper_CNN_LSTM")
    return model

# ==============================================================================
# 3. 데이터 로드 및 전처리 (Physics -> AI Data)
# ==============================================================================
def load_and_preprocess_data():
    file_list = glob.glob(DATA_FILE_PATTERN)
    if not file_list:
        raise FileNotFoundError("❌ 데이터 파일(*.npy)을 찾을 수 없습니다. 시뮬레이션을 먼저 실행하세요.")
    
    print(f"📂 발견된 데이터 파일: {len(file_list)}개")
    
    all_bursts_H = [] # 변환된 H 행렬들을 담을 리스트

    for fname in file_list:
        print(f"Reading {fname}...")
        data = np.load(fname, allow_pickle=True)
        
        for burst in tqdm(data, desc=f"Processing {fname}"):
            a_list = burst['a']     # (30, 1, 8, 3, 8, Paths)
            tau_list = burst['tau'] # (30, 1, 3, Paths) -> Sionna 구조에 따라 다름
            
            # Burst 내 30개 스냅샷 처리
            h_seq = []
            for i in range(len(a_list)):
                # 1. Path Gain (a) 처리
                # 예상 Shape: (1, Rx=8, Tx=3, TxAnt=8, Paths)
                # 목표 Shape: (Rx=8, TxTotal=24, Paths)
                if a_list[i].size == 0:
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue
                    
                a_raw = a_list[i] 
                # 차원 축소 및 병합: (1, 8, 3, 8, Paths) -> (8, 3, 8, Paths) -> (8, 24, Paths)
                # squeeze로 불필요한 차원 제거 (Batch 등)
                try:
                    a_sq = a_raw.reshape(8, 3, 8, -1) 
                    a_flat = a_sq.reshape(8, 24, -1) # (Rx, Tx*TxAnt, Paths)
                except:
                    # Shape이 안 맞을 경우 예외 처리 (빈 데이터 등)
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue

                # 2. Delay (tau) 처리
                tau_raw = tau_list[i]
                # Broadcasting을 위해 차원 맞추기
                # tau는 보통 (Batch, Tx, Paths) 형태임 -> (1, 3, Paths)
                # 이를 (8, 24, Paths)로 확장해야 함
                try:
                    tau_sq = tau_raw.flatten() # 일단 펼침
                    # Paths 개수가 a와 맞는지 확인 필요하지만, 여기선 Broadcasting 이용
                    # 가장 간단한 방법: tau를 (1, 1, Paths)로 보고 확장
                    # 하지만 정확히는 Tx별로 다르므로, (3, Paths) -> (24, Paths) -> (8, 24, Paths)
                    
                    # 간단화: tau shape의 마지막 차원이 Paths라고 가정
                    num_paths = a_flat.shape[-1]
                    tau_flat = tau_raw.reshape(-1) # 전체 다 펼치고
                    # 경로 개수에 맞춰 자르거나 확장 (Sionna 구조 특성상 복잡하므로 단순화)
                    
                    # [중요] Sionna RT 출력 구조상, tau는 (Tx, Paths) 혹은 (1, Paths)일 수 있음.
                    # 여기서는 계산 효율을 위해 a_flat에 맞는 차원으로 강제 확장
                    tau_broadcast = tau_raw.reshape(1, 1, -1) if tau_raw.ndim == 1 else tau_raw
                    # (Broadcasting은 numpy가 알아서 처리하도록 유도)
                    
                except:
                    h_seq.append(np.zeros((8, 24, 64), dtype=np.complex64))
                    continue

                # 3. Frequency Response 계산: H = sum( a * exp(-j2pi * tau * f) )
                # Frequencies: (1, 1, 1, 64)
                # a_flat: (8, 24, Paths, 1)
                # tau: (..., Paths, 1)
                
                f_reshaped = FREQUENCIES.reshape(1, 1, 1, -1)
                a_input = a_flat[..., np.newaxis] 
                
                # tau 차원 맞추기 (약식: 경로 수만 맞으면 작동)
                if tau_raw.size > 0:
                    # tau의 마지막 차원이 Paths라고 가정
                    tau_input = tau_raw.flatten()[:a_flat.shape[-1]] # 경로 수 맞춤
                    tau_input = tau_input.reshape(1, 1, -1, 1)
                    
                    phase = -1j * 2 * np.pi * tau_input * f_reshaped
                    h_val = np.sum(a_input * np.exp(phase), axis=-2) # Sum over paths
                else:
                    h_val = np.zeros((8, 24, 64), dtype=np.complex64)

                h_seq.append(h_val) # (8, 24, 64)

            all_bursts_H.append(np.array(h_seq)) # (30, 8, 24, 64)

    return all_bursts_H

def create_dataset(bursts_data):
    X, Y = [], []
    print("🔄 데이터셋(X, Y) 생성 중...")
    
    for burst in bursts_data:
        # burst shape: (30, 8, 24, 64) (Complex)
        if len(burst) < INPUT_SEQ_LEN + PRED_SEQ_LEN:
            continue
            
        # 복소수 -> 실수/허수 채널 분리 (8, 24, 64) -> (8, 24, 128)
        # Real: [..., 0:64], Imag: [..., 64:128]
        burst_real = burst.real
        burst_imag = burst.imag
        burst_concat = np.concatenate([burst_real, burst_imag], axis=-1) # (30, 8, 24, 128)
        
        # 슬라이딩 윈도우
        for i in range(len(burst) - INPUT_SEQ_LEN - PRED_SEQ_LEN + 1):
            X.append(burst_concat[i : i+INPUT_SEQ_LEN])
            Y.append(burst_concat[i+INPUT_SEQ_LEN : i+INPUT_SEQ_LEN+PRED_SEQ_LEN])
            
    return np.array(X), np.array(Y)

# ==============================================================================
# 4. 메인 실행 (학습)
# ==============================================================================
if __name__ == "__main__":
    # 1. GPU 확인
    strategy = tf.distribute.MirroredStrategy()
    print(f"✅ 사용 가능한 GPU 수: {strategy.num_replicas_in_sync}")

    # 2. 데이터 로드 및 변환
    bursts = load_and_preprocess_data()
    if len(bursts) == 0:
        print("❌ 데이터가 비어있습니다.")
        exit()
        
    # 3. 학습 데이터셋 만들기
    X_train, Y_train = create_dataset(bursts)
    print(f"📊 학습 데이터 준비 완료:")
    print(f"   - X shape: {X_train.shape} (Samples, Time, Rx, Tx, Feat)")
    print(f"   - Y shape: {Y_train.shape}")
    
    # 4. 모델 생성 및 학습
    with strategy.scope():
        model = build_paper_cnn_lstm(INPUT_SEQ_LEN, PRED_SEQ_LEN)
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        
        print("\n🚀 학습 시작 (Training)...")
        history = model.fit(
            X_train, Y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=VALIDATION_SPLIT,
            verbose=1
        )
        
    # 5. 모델 저장
    model.save("cnn_lstm_mimo_model.h5")
    print("💾 모델 저장 완료: cnn_lstm_mimo_model.h5")

2025-12-31 11:14:29.328324: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-31 11:14:29.340617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767147269.355206 2139516 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767147269.359463 2139516 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-31 11:14:29.374120: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3')


I0000 00:00:1767147271.709680 2139516 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22266 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:19:00.0, compute capability: 8.6
I0000 00:00:1767147271.711097 2139516 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22267 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:1a:00.0, compute capability: 8.6
I0000 00:00:1767147271.712350 2139516 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 22267 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:67:00.0, compute capability: 8.6
I0000 00:00:1767147271.713677 2139516 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 21706 MB memory:  -> device: 3, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:68:00.0, compute capability: 8.6


✅ 사용 가능한 GPU 수: 4
📂 발견된 데이터 파일: 4개
Reading channel_history_burst_mimo_3bs_part2.npy...


Processing channel_history_burst_mimo_3bs_part2.npy: 100%|██████████| 250/250 [00:02<00:00, 83.68it/s]


...

138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.9003e-10 - mae: 1.5029e-05 - val_loss: 3.6553e-10 - val_mae: 1.2908e-05
Epoch 18/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.8635e-10 - mae: 1.4994e-05 - val_loss: 1.4906e-09 - val_mae: 2.9838e-05
Epoch 19/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.2555e-10 - mae: 1.4222e-05 - val_loss: 1.2381e-09 - val_mae: 2.7254e-05
Epoch 20/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.6287e-10 - mae: 1.4705e-05 - val_loss: 4.3526e-10 - val_mae: 1.5181e-05
Epoch 21/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.1109e-10 - mae: 1.3938e-05 - val_loss: 3.8066e-09 - val_mae: 4.7534e-05
Epoch 22/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.7347e-10 - mae: 1.5056e-05 - val_loss: 3.8111e-10 - val_mae: 1.3678e-05
Epoch 23/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.6363e-10 - mae: 1.4826e-05 - val_loss: 3.0705e-09 - val_mae: 4.2711e-05
Epoch 24/100
138/138 ━━━━━━━━━━━━━━━━━━━

2025-12-31 11:28:42.375290: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]


138/138 ━━━━━━━━━━━━━━━━━━━━ 19s 133ms/step - loss: 4.4714e-10 - mae: 1.4786e-05 - val_loss: 3.7229e-10 - val_mae: 1.3449e-05
Epoch 34/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.6720e-10 - mae: 1.5076e-05 - val_loss: 6.5256e-10 - val_mae: 1.9501e-05
Epoch 35/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.0134e-10 - mae: 1.3852e-05 - val_loss: 3.9441e-10 - val_mae: 1.4147e-05
Epoch 36/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.9216e-10 - mae: 1.5724e-05 - val_loss: 3.7896e-10 - val_mae: 1.3443e-05
Epoch 37/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.7125e-10 - mae: 1.5085e-05 - val_loss: 1.3104e-09 - val_mae: 2.8014e-05
Epoch 38/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.1142e-10 - mae: 1.4172e-05 - val_loss: 7.7277e-10 - val_mae: 2.1403e-05
Epoch 39/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.5524e-10 - mae: 1.5204e-05 - val_loss: 3.6394e-10 - val_mae: 1.3000e-05
Epoch 40/100
138/138 ━━━━━━━━━━━━━━━━━━━

2025-12-31 11:38:28.295631: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]


138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.3199e-10 - mae: 1.4658e-05 - val_loss: 4.3225e-10 - val_mae: 1.5116e-05
Epoch 66/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.4165e-10 - mae: 1.4797e-05 - val_loss: 2.0841e-09 - val_mae: 3.5361e-05
Epoch 67/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.2770e-10 - mae: 1.4392e-05 - val_loss: 4.0107e-10 - val_mae: 1.4177e-05
Epoch 68/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.2674e-10 - mae: 1.4433e-05 - val_loss: 4.0707e-10 - val_mae: 1.4450e-05
Epoch 69/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 129ms/step - loss: 4.3796e-10 - mae: 1.4790e-05 - val_loss: 4.2403e-10 - val_mae: 1.4961e-05
Epoch 70/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 129ms/step - loss: 4.1916e-10 - mae: 1.4325e-05 - val_loss: 5.5359e-10 - val_mae: 1.7742e-05
Epoch 71/100
138/138 ━━━━━━━━━━━━━━━━━━━━ 18s 130ms/step - loss: 4.2225e-10 - mae: 1.4445e-05 - val_loss: 1.1019e-09 - val_mae: 2.5682e-05
Epoch 72/100
138/138 ━━━━━━━━━━━━━━━━━━━